# ENTSO-E Data Preprocessing

**Time Index**

* Convert all time indices to UTC before any other step.
* Resample all datasets to a common hourly UTC grid (mean of 15-min values), which also turns missing timestamps into missing values.

**Missing values**

* Zeros in `generation_forecast` are treated as missing values before any gap is filled.
* Gaps of up to 3 hours – linear interpolation: the missing 30 November 00:00 price timestamp in all non-PL zones, DST-related gaps in DE_LU and the isolated one-hour gaps in LT `load_forecast` and LT `generation_wind_solar_forecast`.
* Longer gaps in weather-driven series (`generation_wind_solar_forecast`) – copy the same hours from the previous available day (D-1): the seven whole-day gaps in SE_4 (for the 2-day gap use the last available day before it).
* Longer gaps in all other series (demand- and schedule-driven) – copy the same hours from one week earlier (D-7): SE_4 `load_forecast` (2025-12-04) and `generation_forecast` (2025-12-02), SK `generation_forecast` (2023-12-08) and LT `load_forecast` (2026-10-01).
* D-1 is used for weather-driven series because weather persists from one day to the next, and D-7 for demand- and schedule-driven series because they follow a weekly (weekday/weekend) cycle. If the source day is missing as well, the next earlier day is used. The 3-hour limit keeps interpolation within a small part of the daily cycle (solar, load); the gaps found are either at most one missing hour or whole days, so no case lies close to the limit.
* PL: no gaps to fill, only the UTC conversion and hourly resampling change the data.

In [1]:
import pandas as pd
import logging
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent.parent.parent))

from src.config import (
    DATA_DIR,
    DOWNLOAD_START_DATE,
    DOWNLOAD_END_DATE,
)
from src.utils.data_quality import print_missing_values, print_index_summary

from src.utils.preprocessing import preprocess_entsoe, merge_entsoe_zones
from src.utils.file_io import (
    inspect_entsoe_zone_files,
    save_entsoe_zones,
    load_entsoe_data,
)

In [2]:
date_range = f"{DOWNLOAD_START_DATE}_{DOWNLOAD_END_DATE}"
logging.basicConfig(level=logging.INFO)

In [3]:
data_entsoe = load_entsoe_data(
    DATA_DIR / "clean" / "entsoe",
    date_range,
)

In [4]:
data_entsoe = preprocess_entsoe(data_entsoe)

## 1. Index Integrity – Confirmation

All datasets in all bidding zones now share a UTC index with a single 1-hour step (38,279 steps), without duplicated timestamps and without Summer/Winter time repeats, so the gaps in the time index are gone. The only difference is that `day_ahead_prices` ends one hour later than the other datasets (38,280 steps), at 2026-10-21 00:00 local time.

In [5]:
for code, datasets in data_entsoe.items():
    print(f"===== {code} {"=" * 20}")
    for name, data in datasets.items():
        print_index_summary(data=data, name=name)

===== CZ ====================

--- [load_forecast] Index summary ---
Duplicates: 0
Duplicates due to Summer/Winter time switch: 0
Timezone:   UTC

Time steps:
  0 days 01:00:00: 38279

--- [day_ahead_prices] Index summary ---
Duplicates: 0
Duplicates due to Summer/Winter time switch: 0
Timezone:   UTC

Time steps:
  0 days 01:00:00: 38280

--- [generation_wind_solar_forecast] Index summary ---
Duplicates: 0
Duplicates due to Summer/Winter time switch: 0
Timezone:   UTC

Time steps:
  0 days 01:00:00: 38279

--- [generation_forecast] Index summary ---
Duplicates: 0
Duplicates due to Summer/Winter time switch: 0
Timezone:   UTC

Time steps:
  0 days 01:00:00: 38279
===== DE_LU ====================

--- [load_forecast] Index summary ---
Duplicates: 0
Duplicates due to Summer/Winter time switch: 0
Timezone:   UTC

Time steps:
  0 days 01:00:00: 38279

--- [day_ahead_prices] Index summary ---
Duplicates: 0
Duplicates due to Summer/Winter time switch: 0
Timezone:   UTC

Time steps:
  0 days 

## 2. Missing Values – Confirmation

No missing values remain in any dataset: all gaps were filled according to the strategies above. The one exception is the CZ `generation_forecast` column `Scheduled Consumption`, which is empty because it contains only zeros in the source. It is not used in the feature set.

In [6]:
for code, datasets in data_entsoe.items():
    print(f"===== {code} {"=" * 20}")
    for name, data in datasets.items():
        print_missing_values(data=data, name=name)

===== CZ ====================

--- [load_forecast] Missing values ---
No missing values.

--- [day_ahead_prices] Missing values ---
No missing values.

--- [generation_wind_solar_forecast] Missing values ---
No missing values.

--- [generation_forecast] Missing values ---
Scheduled Consumption    38280
===== DE_LU ====================

--- [load_forecast] Missing values ---
No missing values.

--- [day_ahead_prices] Missing values ---
No missing values.

--- [generation_wind_solar_forecast] Missing values ---
No missing values.

--- [generation_forecast] Missing values ---
No missing values.
===== LT ====================

--- [load_forecast] Missing values ---
No missing values.

--- [day_ahead_prices] Missing values ---
No missing values.

--- [generation_wind_solar_forecast] Missing values ---
No missing values.
===== PL ====================

--- [load] Missing values ---
No missing values.

--- [generation] Missing values ---
No missing values.

--- [load_forecast] Missing values --

## 3. Conclusions

The preprocessing decisions from the data quality audit have been applied to all bidding zones (PL, CZ, DE_LU, LT, SE_4 and SK), and the checks above confirm the result. The preprocessed datasets are saved to `data/preprocessed/entsoe/` as one merged parquet per bidding zone and verified by reading the files back. Each file covers 2022-06-08 22:00 to 2026-10-20 22:00 UTC; in the last hour only `day_ahead_prices` has a value.

## 4. Saving Merged Data

In [7]:
merged_entsoe = merge_entsoe_zones(data_entsoe)
merged_dir = DATA_DIR / "preprocessed" / "entsoe" / date_range
save_entsoe_zones(merged_entsoe, merged_dir)
inspect_entsoe_zone_files(merged_dir)

INFO:src.utils.file_io:Saved preprocessed/entsoe/2022-06-09_2026-10-21/CZ.parquet
INFO:src.utils.file_io:Saved preprocessed/entsoe/2022-06-09_2026-10-21/DE_LU.parquet
INFO:src.utils.file_io:Saved preprocessed/entsoe/2022-06-09_2026-10-21/LT.parquet
INFO:src.utils.file_io:Saved preprocessed/entsoe/2022-06-09_2026-10-21/PL.parquet
INFO:src.utils.file_io:Saved preprocessed/entsoe/2022-06-09_2026-10-21/SE_4.parquet
INFO:src.utils.file_io:Saved preprocessed/entsoe/2022-06-09_2026-10-21/SK.parquet



--- ENTSO-E zones (6 file/s) ------------------------------
  [FILE] data/preprocessed/entsoe/2022-06-09_2026-10-21/CZ.parquet | shape: (38281, 5) | idx: 2022-06-08 22:00:00+00:00-2026-10-20 22:00:00+00:00 | cols: 5 | NaN: 38284
  [FILE] data/preprocessed/entsoe/2022-06-09_2026-10-21/DE_LU.parquet | shape: (38281, 6) | idx: 2022-06-08 22:00:00+00:00-2026-10-20 22:00:00+00:00 | cols: 6 | NaN: 5
  [FILE] data/preprocessed/entsoe/2022-06-09_2026-10-21/LT.parquet | shape: (38281, 4) | idx: 2022-06-08 22:00:00+00:00-2026-10-20 22:00:00+00:00 | cols: 4 | NaN: 3
  [FILE] data/preprocessed/entsoe/2022-06-09_2026-10-21/PL.parquet | shape: (38281, 17) | idx: 2022-06-08 22:00:00+00:00-2026-10-20 22:00:00+00:00 | cols: 17 | NaN: 16
  [FILE] data/preprocessed/entsoe/2022-06-09_2026-10-21/SE_4.parquet | shape: (38281, 5) | idx: 2022-06-08 22:00:00+00:00-2026-10-20 22:00:00+00:00 | cols: 5 | NaN: 4
  [FILE] data/preprocessed/entsoe/2022-06-09_2026-10-21/SK.parquet | shape: (38281, 3) | idx: 2022-06-